# 2차 — LightGBM LambdaRank 손실함수 실험

## 배경
- 현재 OOF: 단일 LightGBM(binary) 0.7400~0.74053, 멀티시드 배깅 0.74036~0.74037, 팀 블렌딩 0.74071
- 멀티시드 견고탐색까지 시도했지만 단일시드 탐색과 거의 차이 없음(+0.00001) → **하이퍼파라미터 탐색 방법론 쪽은 거의 천장**
- 평가지표가 ROC-AUC(순위 기반)인데 지금까지는 전부 binary logloss로 학습 → 아직 안 건드려본 축은 **손실함수 자체**

## 핵심 아이디어
- `objective='lambdarank'` + `metric='auc'`로 학습 중에도 실제 평가지표를 직접 모니터링
- LightGBM lambdarank는 query(그룹) 하나당 행 수가 10,000개를 못 넘는 하드캡이 있음(pairwise 비교가 O(n²)이라 안전장치) → fold 전체를 한 그룹으로 묶지 않고, 셔플 후 작은 인공 그룹(기본 100행)으로 잘라서 그 안에서만 비교하도록 구성
- lambdarank 출력은 보정된 확률이 아니라 임의 스케일의 순위 점수이므로, **fold별로 즉시 rank-정규화([0,1] 백분위)** 해서 합쳐야 OOF AUC 계산과 다른 모델과의 블렌딩이 의미 있음

## 전처리 / 파라미터 출처
- 전처리는 `2차_lgbm_robust_multiseed_search` 노트북의 실제 코드를 그대로 가져옴 (TARGET_COL="임신 성공 여부", DEAD_COLS 2개 드롭, `is_DI`/`froze_embryo` 파생, 결측치 처리, LabelEncoder)
  - 차이점: 그 노트북은 train만 처리했지만(CV 실험), 여기는 test 예측도 필요해서 **LabelEncoder를 train+test 합쳐서 fit** — xgboost에서 겪었던 카테고리 불일치 크래시와 같은 함정을 미리 차단
- 시작 파라미터는 멀티시드 탐색에서 채택된 `lgbm_robust_best_params.json` (sklearn API 이름이지만 LightGBM이 공식 alias로 인식: subsample→bagging_fraction, colsample_bytree→feature_fraction, reg_alpha→lambda_l1, reg_lambda→lambda_l2, min_child_samples→min_data_in_leaf, min_split_gain→min_gain_to_split). `n_estimators`만 `num_boost_round`와 충돌 가능성이 있어 따로 분리해서 처리.

---
**저장 위치:** `experiment_history/2차/2차_lgbm_lambdarank.ipynb` (멀티시드 탐색 노트북과 같은 폴더)


In [10]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score


In [11]:
# 노트북 위치: experiment_history/2차/ (멀티시드 탐색 노트북과 동일 폴더)
DATA_DIR = Path("../../data")
SHARED_DIR = Path("..")              # blend_cache, *_best_params.json 등 공유 자원
SUBMIT_DIR = Path("../submit file")  # 실험 중 생성되지만 실제 제출은 안 한 CSV

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]

# 멀티시드 탐색에서 채택된 파라미터를 기본값으로. 없으면 기존 best_params.json으로 자동 폴백
ROBUST_PARAMS_PATH = SHARED_DIR / "lgbm_robust_best_params.json"
FALLBACK_PARAMS_PATH = SHARED_DIR / "best_params.json"

N_SPLITS = 5
SEED = 42
EARLY_STOPPING_ROUNDS = 100
N_BOOST_ROUND = 3000  # 충분히 크게, early stopping이 실제 라운드 수를 결정

# LightGBM lambdarank는 query(그룹) 하나당 행 수가 10000개를 못 넘음(하드캡, pairwise 비교가 O(n^2)이라
# 안전장치로 막아둔 것). fold 전체(~20만 행)를 한 그룹으로 묶으면 에러 나므로, 작은 인공 그룹으로 잘라서
# 그 안에서만 pairwise 비교를 함. 너무 작으면 그룹 안에 양성 샘플이 없어 그래디언트가 0이 될 수 있고,
# 너무 크면 느려짐(O(group_size^2)) — 100 정도가 균형점.
GROUP_SIZE = 100


## 1. 데이터 로드 + 전처리 (멀티시드 탐색 노트북과 동일 로직, train+test 둘 다 처리)

In [12]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])
test_raw_full = pd.read_csv(TEST_PATH)
test_ids = test_raw_full["ID"]
test_raw = test_raw_full.drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df_train = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
df_test = fill_na(base_features(test_raw.copy()).drop(columns=DEAD_COLS))

cat_cols = df_train.select_dtypes(include=["object"]).columns.tolist()

# train+test 합쳐서 LabelEncoder fit (카테고리 불일치로 인한 크래시 방지 — xgboost에서 겪은 것과 동일 패턴)
df_train_le = df_train.copy()
df_test_le = df_test.copy()
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([df_train_le[col], df_test_le[col]], axis=0).astype(str)
    le.fit(combined)
    df_train_le[col] = le.transform(df_train_le[col].astype(str))
    df_test_le[col] = le.transform(df_test_le[col].astype(str))

y = df_train_le[TARGET_COL].values.astype(np.float32)
X = df_train_le.drop(columns=[TARGET_COL])
X_test = df_test_le.copy()

print(f"전처리 완료: train {X.shape}, test {X_test.shape}")


/var/folders/s7/xbnkmtj92j5_ly_bxspy44f40000gn/T/ipykernel_93497/2307222159.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object"]).columns
/var/folders/s7/xbnkmtj92j5_ly_bxspy44f40000gn/T/ipykernel_93497/2307222159.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://p

전처리 완료: train (256351, 67), test (90067, 67)


## 2. CV 설정 + 시작 파라미터 로드

In [13]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))
print(f"{N_SPLITS}-fold 분할 완료 (seed={SEED})")


5-fold 분할 완료 (seed=42)


In [14]:
def load_base_params(path: Path) -> dict:
    with open(path, encoding="utf-8") as f:
        loaded = json.load(f)
    return loaded.get("best_params", loaded) if isinstance(loaded, dict) else loaded

params_path = ROBUST_PARAMS_PATH if ROBUST_PARAMS_PATH.exists() else FALLBACK_PARAMS_PATH
base_params = load_base_params(params_path)
print(f"파라미터 출처: {params_path}")

# n_estimators는 num_boost_round(함수 인자)와 역할이 겹쳐서 분리, 나머지 sklearn식 이름은
# LightGBM이 공식 alias로 인식하므로 그대로 둠 (subsample→bagging_fraction 등)
base_params.pop("n_estimators", None)

# binary objective 전용 키 제거 (혹시 들어있다면 — 어차피 효과 없었던 레버였음)
for k in ["objective", "metric", "is_unbalance", "scale_pos_weight"]:
    base_params.pop(k, None)

params = {
    **base_params,
    "objective": "lambdarank",
    "metric": "auc",        # objective와 무관하게 실제 평가지표를 직접 모니터링
    "label_gain": [0, 1],   # 이진 레이블(0/1)이므로 명시
    "random_state": SEED,
    "verbosity": -1,
}

print(json.dumps(params, indent=2, ensure_ascii=False))


파라미터 출처: ../lgbm_robust_best_params.json
{
  "learning_rate": 0.03735839746471122,
  "num_leaves": 253,
  "max_depth": 5,
  "min_child_samples": 41,
  "subsample": 0.699825092698368,
  "colsample_bytree": 0.7084506083750325,
  "reg_alpha": 3.9529019873195606,
  "reg_lambda": 5.86734589907409,
  "min_split_gain": 0.5142456412311088,
  "objective": "lambdarank",
  "metric": "auc",
  "label_gain": [
    0,
    1
  ],
  "random_state": 42,
  "verbosity": -1
}


## 3. fold별 학습 (핵심)

In [15]:
def rank_normalize(scores: np.ndarray) -> np.ndarray:
    """fold마다 임의 스케일인 lambdarank 점수를 [0,1] 백분위로 정규화"""
    return rankdata(scores) / len(scores)


def make_group_sizes(n: int, group_size: int) -> list:
    """n개 행을 group_size 단위로 잘라서 그룹 크기 리스트를 만듦 (마지막 그룹은 나머지)"""
    sizes = [group_size] * (n // group_size)
    remainder = n % group_size
    if remainder:
        sizes.append(remainder)
    return sizes


def shuffle_rows(X_part: pd.DataFrame, y_part: np.ndarray, seed: int):
    """인공 그룹을 만들기 전에 행 순서를 섞음 (그룹마다 0/1이 적절히 섞이도록)"""
    rng = np.random.RandomState(seed)
    order = rng.permutation(len(X_part))
    return X_part.iloc[order].reset_index(drop=True), y_part[order]


oof = np.zeros(len(X))
test_fold_preds = []  # fold별 rank-정규화된 test 예측을 모아둠
fold_aucs = []

start = time.time()
for fold, (tr_idx, val_idx) in enumerate(folds):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    # 그룹 구성용으로만 섞은 버전 (학습/검증용 그래디언트·메트릭 계산에만 사용)
    X_tr_shuf, y_tr_shuf = shuffle_rows(X_tr, y_tr, seed=SEED + fold)
    X_val_shuf, y_val_shuf = shuffle_rows(X_val, y_val, seed=SEED + fold + 1000)

    train_group = make_group_sizes(len(X_tr_shuf), GROUP_SIZE)
    valid_group = make_group_sizes(len(X_val_shuf), GROUP_SIZE)

    train_set = lgb.Dataset(X_tr_shuf, label=y_tr_shuf, group=train_group)
    valid_set = lgb.Dataset(X_val_shuf, label=y_val_shuf, group=valid_group, reference=train_set)

    model = lgb.train(
        params,
        train_set,
        num_boost_round=N_BOOST_ROUND,
        valid_sets=[valid_set],
        callbacks=[
            lgb.early_stopping(EARLY_STOPPING_ROUNDS),
            lgb.log_evaluation(100),
        ],
    )

    # 예측은 원래 순서 그대로의 X_val/X_test에 대해 수행 — group은 학습시 그래디언트 계산에만
    # 쓰이고, predict()는 row 순서/그룹과 무관하게 각 행의 점수를 그대로 출력함
    val_raw = model.predict(X_val, num_iteration=model.best_iteration)
    fold_auc = roc_auc_score(y_val, val_raw)  # rank-정규화는 단조변환이라 fold 내부 AUC는 그대로
    fold_aucs.append(fold_auc)
    oof[val_idx] = rank_normalize(val_raw)

    test_raw = model.predict(X_test, num_iteration=model.best_iteration)
    test_fold_preds.append(rank_normalize(test_raw))

    print(f"[fold {fold}] best_iter={model.best_iteration}, AUC={fold_auc:.5f}")

print(f"\n총 소요 시간: {time.time() - start:.1f}s")


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.732567
[200]	valid_0's auc: 0.734424
[300]	valid_0's auc: 0.734432
Early stopping, best iteration is:
[205]	valid_0's auc: 0.734432
[fold 0] best_iter=205, AUC=0.73443
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.736053
[200]	valid_0's auc: 0.738101
Early stopping, best iteration is:
[196]	valid_0's auc: 0.738105
[fold 1] best_iter=196, AUC=0.73810
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.734756
[200]	valid_0's auc: 0.736307
[300]	valid_0's auc: 0.736312
Early stopping, best iteration is:
[206]	valid_0's auc: 0.736312
[fold 2] best_iter=206, AUC=0.73631
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.733358
[200]	valid_0's auc: 0.734825
[300]	valid_0's auc: 0.73484
Early stopping, best iteration is:
[215]	valid_0's auc: 0.73484
[fold 3] best_iter=215, AUC=0.73484
Training until valid

## 4. OOF 결과 확인

In [16]:
oof_auc = roc_auc_score(y, oof)
print(f"Fold별 AUC: {[round(a, 5) for a in fold_aucs]}")
print(f"Fold AUC 평균±표준편차: {np.mean(fold_aucs):.5f} ± {np.std(fold_aucs):.5f}")
print(f"전체 OOF AUC (rank-정규화 기준): {oof_auc:.5f}")
print()
print("비교 기준:")
print("  기존 binary LightGBM 단일시드 OOF: 0.7400 ~ 0.74053")
print("  멀티시드 배깅 OOF: 0.74036 (기존) / 0.74037 (견고탐색)")
print("  팀 블렌딩 OOF: 0.74071")


Fold별 AUC: [0.73443, 0.7381, 0.73631, 0.73484, 0.73628]
Fold AUC 평균±표준편차: 0.73599 ± 0.00130
전체 OOF AUC (rank-정규화 기준): 0.73599

비교 기준:
  기존 binary LightGBM 단일시드 OOF: 0.7400 ~ 0.74053
  멀티시드 배깅 OOF: 0.74036 (기존) / 0.74037 (견고탐색)
  팀 블렌딩 OOF: 0.74071


## 5. test 예측 평균 + 저장

In [17]:
test_pred = np.mean(test_fold_preds, axis=0)  # 이미 fold별로 rank-정규화 했으므로 단순 평균 가능

BLEND_DIR = SHARED_DIR / "blend_cache"
BLEND_DIR.mkdir(exist_ok=True)
np.save(BLEND_DIR / "lgbm_lambdarank_oof.npy", oof)
np.save(BLEND_DIR / "lgbm_lambdarank_test.npy", test_pred)
print(f"저장 완료: {BLEND_DIR / 'lgbm_lambdarank_oof.npy'}, {BLEND_DIR / 'lgbm_lambdarank_test.npy'}")

# 실험 단계 비제출 CSV (제출 여부는 OOF 결과 보고 판단)
SUBMIT_DIR.mkdir(exist_ok=True)
submission = pd.DataFrame({"ID": test_ids, "probability": test_pred})
out_path = SUBMIT_DIR / f"2차_lgbm_lambdarank_local{oof_auc:.5f}.csv"
submission.to_csv(out_path, index=False)
print(f"비제출용 결과 저장: {out_path}")


저장 완료: ../blend_cache/lgbm_lambdarank_oof.npy, ../blend_cache/lgbm_lambdarank_test.npy
비제출용 결과 저장: ../submit file/2차_lgbm_lambdarank_local0.73599.csv


## 6. 결과 해석 & 다음 단계

- **OOF AUC가 0.74037(멀티시드 배깅 최고치)보다 확실히 높다면**: 손실함수 변경이 실제로 먹힌 것 →
  `blend_cache/lgbm_lambdarank_oof.npy`를 팀 블렌딩 노트북에 추가해볼 가치 있음.
- **거의 같거나 낮다면**: 이 데이터셋은 손실함수를 바꿔도 같은 천장(~0.74)에 도달한다는 추가 증거
  (모델 다양화, 멀티시드 견고탐색과 같은 패턴) — 굳이 채택할 필요 없음.
- 결과가 promising하면, 멀티시드 탐색 노트북의 **10시드 배깅 패턴**을 그대로 이 lambdarank 학습에
  적용해서(현재는 단일 seed=42, 5-fold) 한 번 더 검증해볼 수 있음. 지금은 빠른 1차 판단을 위해
  단일 시드로만 돌렸습니다.
- XGBoost `rank:pairwise`도 같은 group 구성 + rank-정규화 패턴으로 시도 가능
  (기존 `xgboost_best_params.json` 재사용).
